A little throwback to college days, but with a lot more learning! :eyes:

The IIT Roorkee Campus Immersion 2026 is slowly coming together, and we’re super excited about everything we have planned for you! :sparkles:
We’ve started connecting with learners gradually, and over the next few days, our team will be reaching out to more of you to understand your interest and help you with the next steps.

But before we get there, we’d love to start the conversation right here! :speech_balloon:

Whether you’re curious about:
:mortar_board: What the campus experience will look like
:school: What we’ll be doing at IIT Roorkee
:bulb: The learning sessions, hackathons & challenges
:handshake: Meeting and collaborating with fellow learners
:male-teacher: Interacting with IIT Roorkee professors & Scaler faculty
:house: Stay, travel, logistics or anything else
:sparkles: Or simply want to know what makes this experience special

Drop your questions here! We’ll keep answering them and sharing more details as we get closer.

And if you haven’t shared your interest yet, you can do that here :point_down:
:point_right: Share your interest:
https://docs.google.com/forms/d/e/1FAIpQLSd5yPquq1J0XYPo1ppGK6OAK6hAK8f1M-HaC9IxTroOligPBg/viewform?usp=sharing&ouid=113200008645481478431

:round_pushpin: IIT Roorkee Campus
:date: 13th–15th November 2026

There’s a lot in the works — learning, building, collaborating, exploring the campus, making new connections and, of course, creating some great memories along the way! :heart:

So, let’s start building the excitement together. :rocket:

What’s the first thing you want to know about the IIT Roorkee experience? :eyes:

# Porter

## One request, start to finish

```mermaid
---
config:
  flowchart:
    wrappingWidth: 360
---
flowchart TD
    B(["a customer's browser<br/>http://…:8080"]) --> MW
    subgraph BOX["the porter container · P1, then on EC2 · P4"]
        MW["measure() · P2<br/>counts and times every request"] --> CH["POST /v1/chat"]
        CH --> AG["build_agent()<br/>the model, three tools, three limits"]
        AG --> T["build_tools()<br/>look_up_order · price_order · start_return"]
        AP["POST /v1/returns/{id}/approve<br/>approve(): one refund, once"]
    end
    T -->|"read"| DB[("online_retail.db<br/>the shop's orders, read-only")]
    T -->|"start_return writes a request"| L[("the ledger<br/>returns · refunds")]
    ST(["a member of staff<br/>/staff"]) --> AP
    AP -->|"writes the refund"| L
    AG <-->|"https + the key"| M["gpt-4.1-mini<br/>on OpenAI's API"]
    PG[("Postgres · P1<br/>the conversations")] <-->|"load · save, by thread_id"| AG
    PR[("Prometheus · P2<br/>reads /metrics every 5 s")] -.-> MW
    GF(["Grafana · P3<br/>three panels"]) -.-> PR

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef tools fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef guard fill:#fce8e6,stroke:#d93025,color:#a50e0e
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333
    classDef person fill:#fff3e0,stroke:#ef6c00,color:#8a3800
    class AG,M model
    class MW,CH,T,AP tools
    class DB,L,PG,PR stored
    class B,GF endpoint
    class ST person
    style BOX fill:none,stroke:#9aa0a6,color:#80868b
```

- **The agent** is the model plus three tools, built for one customer. Two tools only read the orders; `start_return` writes a *request* to the ledger.
- **The conversation is kept in Postgres,** a container of its own, under its `thread_id`: the next question that sends that id carries on from it.
- **Only a member of staff pays a refund,** through a route the agent has no tool for, and approving the same request twice pays once.
- **`measure()`** wraps every request, **Prometheus** reads the numbers it keeps, and **Grafana** draws them.

In [ ]:
# The service's packages (requirements.txt), plus two this notebook uses.
#%pip install -q -r requirements.txt python-dotenv ipython-autotime

In [1]:
# From here on, each cell prints how long it took.
%load_ext autotime

time: 96.2 µs (started: 2026-10-01 20:52:59 +05:30)


In [2]:
# The imports later Parts use, and the key: .env's lines become environment variables.
import os
from concurrent.futures import ThreadPoolExecutor

import httpx                                   # sends HTTP requests from Python
from dotenv import dotenv_values, load_dotenv  # reads NAME=value files: .env, deploy/.state

load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")
print("OPENAI_API_KEY", "is set" if os.getenv("OPENAI_API_KEY") else "is missing: copy .env.example to .env, then add it")

# Docker prints plain text in the cells below: no colours, and no "What's next" tips.
os.environ.update(NO_COLOR="1", DOCKER_CLI_HINTS="false")

OPENAI_API_KEY is set
time: 56.2 ms (started: 2026-10-01 20:53:15 +05:30)


# Part 1


```mermaid
---
config:
  flowchart:
    wrappingWidth: 360
---
flowchart TD
    F["porter/ · requirements.txt · Dockerfile"] -->|"docker build"| S1["stage 1 · build<br/>pip install into /install"]
    S1 -->|"only the installed<br/>packages cross"| IMG[("the image porter:1.0<br/>/install + porter/ · runs as user porter<br/>no orders · no key")]
    IMG -->|"docker run, with nothing else"| X{{"key: None<br/>unable to open database file"}}
    IMG -->|"docker compose -f compose.app.yaml up -d<br/>+ the orders, mounted<br/>+ the key, from .env<br/>+ /state, a volume"| C["the porter container<br/>port 8080"]
    C <-->|"the conversations,<br/>by thread_id"| PG[("the postgres container<br/>started first · its own volume")]
    C --> OK(["curl /healthz · /v1/chat<br/>200 · the answer"])
    C --> CONV(["a follow-up, the same thread_id<br/>it knows the order already"])
    C --> RET(["a return request, approved by staff<br/>one refund, even when approved twice"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef tools fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef guard fill:#fce8e6,stroke:#d93025,color:#a50e0e
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333
    classDef person fill:#fff3e0,stroke:#ef6c00,color:#8a3800
    class F,S1,C tools
    class IMG,PG stored
    class X guard
    class OK,CONV,RET endpoint
```

In [6]:
!docker build --progress plain -t porter:1.0 .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 2.26kB done
#1 DONE 0.0s

#2 resolve image config for docker-image://docker.io/docker/dockerfile:1
#2 DONE 0.4s

#3 docker-image://docker.io/docker/dockerfile:1@sha256:4edf897a3ffa55b89f906fc8cc78afdb3f1834cc9c7083565e611a8a7d5fe99e
#3 resolve docker.io/docker/dockerfile:1@sha256:4edf897a3ffa55b89f906fc8cc78afdb3f1834cc9c7083565e611a8a7d5fe99e done
#3 CACHED

#4 [internal] load metadata for docker.io/library/python:3.12-slim
#4 DONE 0.4s

#5 [internal] load .dockerignore
#5 transferring context: 237B done
#5 DONE 0.0s

#6 [internal] load build context
#6 DONE 0.0s

#7 [build 1/4] FROM docker.io/library/python:3.12-slim@sha256:f77ac9e44ae96ef2c90b8053ea08c31f8be030f824196b0ae4db6d462c84e51f
#7 resolve docker.io/library/python:3.12-slim@sha256:f77ac9e44ae96ef2c90b8053ea08c31f8be030f824196b0ae4db6d462c84e51f done
#7 DONE 0.0s

#6 [internal] load bu

In [7]:
!docker run --rm porter:1.0 python -c "import os; print('key:', os.getenv('OPENAI_API_KEY')); from porter.api import healthz; healthz()"

key: None
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/app/porter/api.py", line 83, in healthz
    con = read_only(settings.db_path)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/app/porter/tools.py", line 17, in read_only
    return sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, check_same_thread=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: unable to open database file
time: 1.53 s (started: 2026-10-01 21:12:06 +05:30)


In [1]:
# Step 1: start what compose.app.yaml describes, in the background: Postgres, then Porter once Postgres is healthy. --progress quiet: only errors, no progress bars.
!docker compose -f compose.app.yaml  up

WARN[0000] volume "porter-state" already exists but was created for project "v2" (expected "last-class"). Use `external: true` to use an existing volume 
WARN[0000] volume "porter-pgdata" already exists but was created for project "v2" (expected "last-class"). Use `external: true` to use an existing volume 
[+] up 1/2
 ✔ Network last-class_default      Created                                  0.0s
 ⠋ Container last-class-postgres-1 Creating                                 0.0s
[+] up 3/3
 ✔ Network last-class_default      Created                                  0.0s
 ✔ Container last-class-postgres-1 Created                                  0.0s
 ✔ Container last-class-porter-1   Created                                  0.0s
Attaching to porter-1, postgres-1
Container last-class-postgres-1 Waiting 
78postgres-1  | 
78


7v View in Docker Desktop   o View Config   l View Logs   w Enable Watch   d Detach878postgres-1  | PostgreSQL Database directory appears to contain a database

# Prometheus